In [ ]:
# Block 1: Imports and Setup

import os
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, saving # Import saving module

import matplotlib.pyplot as plt
import joblib

# --- Configuration ---
INPUT_CSV_PATH = '/content/drive/MyDrive/AML /All/stockprice.csv'  # update if needed

# Output folders (created alongside the CSV)
BASE_DIR = os.path.dirname(os.path.abspath(INPUT_CSV_PATH))
OUTPUT_FIGURES_DIR = os.path.join(BASE_DIR, "figures_with_cluster")
OUTPUT_MODELS_DIR  = os.path.join(BASE_DIR, "models_1")
OUTPUT_RESULTS_DIR = os.path.join(BASE_DIR, "results_")
SCALERS_DIR        = os.path.join(OUTPUT_MODELS_DIR, "scalers_1")

os.makedirs(OUTPUT_FIGURES_DIR, exist_ok=True)
os.makedirs(OUTPUT_MODELS_DIR,  exist_ok=True)
os.makedirs(OUTPUT_RESULTS_DIR, exist_ok=True)
os.makedirs(SCALERS_DIR,        exist_ok=True)

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Model types we’ll train
MODEL_TYPES = ['Transformer', 'LSTM', 'RNN']

print(f"Input CSV : {INPUT_CSV_PATH}")
print(f"Figures   : {OUTPUT_FIGURES_DIR}")
print(f"Models   : {OUTPUT_MODELS_DIR}")
print(f"Results   : {OUTPUT_RESULTS_DIR}")
print(f"Scalers   : {SCALERS_DIR}")

Input CSV : /content/drive/MyDrive/AML /All/stockprice.csv
Figures   : /content/drive/MyDrive/AML /All/figures_with_cluster
Models   : /content/drive/MyDrive/AML /All/models_1
Results   : /content/drive/MyDrive/AML /All/results_
Scalers   : /content/drive/MyDrive/AML /All/models_1/scalers_1


In [ ]:
# Block 2: Load and Inspect Data

df = pd.read_csv(INPUT_CSV_PATH)
df.columns = df.columns.str.strip()

# Handle common date name variants
if 'trading date' not in df.columns:
    for alt in ['trading_date', 'date', 'Trading Date', 'Trading date']:
        if alt in df.columns:
            df['trading date'] = df[alt]
            break

# Parse date + sort
df['trading date'] = pd.to_datetime(df['trading date'], errors='coerce')
df = df.dropna(subset=['trading date']).sort_values(['sector', 'trading date']).reset_index(drop=True)

print("\n--- Dataset Info ---")
print(df.info())
print("\n--- Head ---")
print(df.head())

sectors = df['sector'].dropna().unique().tolist()
print("\nSectors:", sectors)



--- Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4270 entries, 0 to 4269
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   sector        4270 non-null   object        
 1   trading_date  4270 non-null   object        
 2   open          4270 non-null   float64       
 3   high          4270 non-null   float64       
 4   low           4270 non-null   float64       
 5   close         4270 non-null   float64       
 6   volume        4269 non-null   float64       
 7   year          4270 non-null   int64         
 8   month         4270 non-null   int64         
 9   date          4270 non-null   int64         
 10  trading date  4270 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(5), int64(3), object(2)
memory usage: 367.1+ KB
None

--- Head ---
        sector trading_date    open    high     low   close     volume  year  \
0  Engineering   2017-05-02  101.57  10

In [ ]:
# Block 3: Define Time2Vec Layer (for Transformer)

@saving.register_keras_serializable() # Use the imported saving module
class Time2Vec(layers.Layer):
    """
    Minimal Time2Vec: concatenates sin(wa*x+ba) with linear (wb*x+bb) along features.
    Input:  (batch, seq, n_feats)
    Output: (batch, seq, 2*n_feats)
    """
    def __init__(self, kernel_size=1, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = kernel_size

    def build(self, input_shape):
        self.n_feats = int(input_shape[-1])
        self.wa = self.add_weight(shape=(self.n_feats,), initializer="uniform", trainable=True, name="wa")
        self.ba = self.add_weight(shape=(self.n_feats,), initializer="uniform", trainable=True, name="ba")
        self.wb = self.add_weight(shape=(self.n_feats,), initializer="uniform", trainable=True, name="wb")
        self.bb = self.add_weight(shape=(self.n_feats,), initializer="uniform", trainable=True, name="bb")
        super().build(input_shape)

    def call(self, x):
        v1 = tf.math.sin(x * self.wa + self.ba)
        v2 = x * self.wb + self.bb
        return tf.concat([v1, v2], axis=-1)

In [ ]:
# Block 4: Preprocessing Functions

SEQ_LEN  = 8
FEATURES = ['open', 'high', 'low', 'close', 'volume']
RET_COLS = [f'return_{c}' for c in FEATURES]

def make_stationary_and_scale(group):
    """
    10-day MA smoothing -> differencing (returns) -> dropna -> MinMax scale on returns.
    Returns processed df and fitted scaler.
    """
    g = group.copy()
    smooth = g[FEATURES].rolling(window=10, min_periods=1).mean()
    for c in FEATURES:
        g[f'return_{c}'] = smooth[c].diff()
    g = g.dropna().reset_index(drop=True)

    scaler = MinMaxScaler()
    g[RET_COLS] = scaler.fit_transform(g[RET_COLS])
    return g, scaler

def create_sequences(data, seq_len=SEQ_LEN):
    """
    Build (X, y, last_close):
      X uses previous seq_len returns; y is next-step return_close (scalar).
    """
    X, y, last_closes = [], [], []
    for i in range(seq_len, len(data)):
        X.append(data[RET_COLS].iloc[i-seq_len:i].values)  # (seq_len, 5)
        y.append(data['return_close'].iloc[i])              # scalar
        last_closes.append(data['close'].iloc[i-1])         # for reconstructing price
    return np.array(X), np.array(y), np.array(last_closes)

# Process all sectors → dicts and save scalers
scalers = {}
processed_data_dict = {}

for s in sectors:
    sub = df[df['sector'] == s].copy()
    if len(sub) < SEQ_LEN + 5:
        print(f"Skipping sector {s}: too few rows.")
        continue
    processed, scaler = make_stationary_and_scale(sub)
    X, y, last_closes = create_sequences(processed, SEQ_LEN)

    processed_data_dict[s] = {
        'X': X, 'y': y, 'last_closes': last_closes, 'raw_data': processed
    }
    scalers[s] = scaler
    # save scaler for real-world inference
    joblib.dump(scaler, os.path.join(SCALERS_DIR, f"scaler_{s.replace(' ', '_')}.pkl"))

    print(f"{s}: X{X.shape}, y{y.shape}, last_closes{last_closes.shape}")

print("Preprocessing complete.")


Engineering: X(845, 8, 5), y(845,), last_closes(845,)
Fuel & Power: X(845, 8, 5), y(845,), last_closes(845,)
IT Sector: X(844, 8, 5), y(844,), last_closes(844,)
Services & Real Estate: X(845, 8, 5), y(845,), last_closes(845,)
Telecommunication: X(845, 8, 5), y(845,), last_closes(845,)
Preprocessing complete.


In [ ]:
# Block 5: Define Model Architectures (many-to-one)

def build_transformer_model(
    seq_length=SEQ_LEN,
    n_features=len(FEATURES),
    d_model=32, n_heads=2, d_ff=64, dropout=0.1, lr=1e-3
):
    inputs = layers.Input(shape=(seq_length, n_features), name='sequence_input')

    # Time2Vec
    t2v = Time2Vec(kernel_size=1)(inputs)                 # (batch, seq, 2*n_feats)
    feat_proj = layers.Dense(d_model, name='feature_projection')(inputs)
    time_proj = layers.Dense(d_model, name='time_projection')(t2v)
    x = layers.Add(name='feature_time_fusion')([feat_proj, time_proj])  # (batch, seq, d_model)

    # Positional embedding
    positions = tf.range(start=0, limit=seq_length, delta=1)            # (seq,)
    pos_emb = layers.Embedding(input_dim=seq_length, output_dim=d_model, name='pos_encoding')(positions)
    pos_emb = tf.expand_dims(pos_emb, axis=0)                           # (1, seq, d_model)
    x = layers.Add(name='add_pos_encoding')([x, pos_emb])

    # Encoder block
    x_norm = layers.LayerNormalization(epsilon=1e-6, name='ln_pre_attn')(x)
    attn = layers.MultiHeadAttention(num_heads=n_heads, key_dim=d_model, name='mha')(x_norm, x_norm)
    x = layers.Add(name='resid_attn')([x, layers.Dropout(dropout)(attn)])

    x_norm = layers.LayerNormalization(epsilon=1e-6, name='ln_pre_ffn')(x)
    ffn = layers.Dense(d_ff, activation='gelu', name='ffn1')(x_norm)
    ffn = layers.Dense(d_model, name='ffn2')(ffn)
    x = layers.Add(name='resid_ffn')([x, layers.Dropout(dropout)(ffn)])

    # Many-to-one head
    x_last = layers.Lambda(lambda t: t[:, -1, :], name='take_last')(x)
    x_last = layers.Dense(32, activation='relu')(x_last)
    x_last = layers.Dropout(0.1)(x_last)
    outputs = layers.Dense(1, name='out')(x_last)                       # (batch, 1)

    model = models.Model(inputs, outputs, name="Transformer_Stock")
    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss='mse', metrics=['mae'])
    return model

def build_lstm_model(seq_length=SEQ_LEN, n_features=len(FEATURES), lstm_units=64, dropout=0.1, lr=1e-3):
    model = models.Sequential(name="LSTM_Stock")
    model.add(layers.Input(shape=(seq_length, n_features)))
    model.add(layers.LSTM(lstm_units, return_sequences=False))
    model.add(layers.Dense(32, activation='relu'))
    model.add(layers.Dropout(dropout))
    model.add(layers.Dense(1))
    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss='mse', metrics=['mae'])
    return model

def build_rnn_model(seq_length=SEQ_LEN, n_features=len(FEATURES), rnn_units=64, dropout=0.1, lr=1e-3):
    model = models.Sequential(name="RNN_Stock")
    model.add(layers.Input(shape=(seq_length, n_features)))
    model.add(layers.SimpleRNN(rnn_units, return_sequences=False))
    model.add(layers.Dense(32, activation='relu'))
    model.add(layers.Dropout(dropout))
    model.add(layers.Dense(1))
    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss='mse', metrics=['mae'])
    return model

In [ ]:
# Block 6: Train Models (per-sector), Evaluate & Store

def ensure_targets_column(y):
    y = np.asarray(y)
    return y.reshape(-1, 1) if y.ndim == 1 else y

def ts_split(X, y, train_ratio=0.8, val_ratio=0.1):
    n = X.shape[0]
    n_train = max(1, int(n*train_ratio))
    n_val   = max(1, int(n*val_ratio))
    n_test  = max(1, n - n_train - n_val)
    if n_train + n_val + n_test > n:
        n_test = max(1, n - n_train - n_val)
    Xtr, ytr = X[:n_train], y[:n_train]
    Xv,  yv  = X[n_train:n_train+n_val], y[n_train:n_train+n_val]
    Xte, yte = X[n_train+n_val:], y[n_train+n_val:]
    return Xtr, ytr, Xv, yv, Xte, yte

def build_model_by_name(name, seq_len, n_feats):
    if name == 'Transformer': return build_transformer_model(seq_len, n_feats)
    if name == 'LSTM':       return build_lstm_model(seq_len, n_feats)
    if name == 'RNN':        return build_rnn_model(seq_len, n_feats)
    raise ValueError(name)

EPOCHS = 50
BATCH_SIZE = 32
EARLY_STOP = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

all_results = {m: {} for m in MODEL_TYPES}
model_histories = {m: {} for m in MODEL_TYPES}

for sector in sectors:
    if sector not in processed_data_dict:
        continue
    print(f"\n===== Sector: {sector} =====")
    pack = processed_data_dict[sector]
    X, y = pack['X'], ensure_targets_column(pack['y'])
    last_closes = pack['last_closes']
    seq_len, n_feats = X.shape[1], X.shape[2]

    Xtr, ytr, Xv, yv, Xte, yte = ts_split(X, y, 0.8, 0.1)
    last_closes_test = last_closes[len(ytr)+len(yv):]

    for model_type in MODEL_TYPES:
        print(f"\n--- {model_type} ---")
        model = build_model_by_name(model_type, seq_len, n_feats)

        hist = model.fit(
            Xtr, ytr,
            validation_data=(Xv, yv),
            epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1,
            callbacks=[EARLY_STOP]
        )
        model_histories[model_type][sector] = hist

        # Evaluate on TEST (normalized)
        test_mse, test_mae = model.evaluate(Xte, yte, verbose=0)

        # Inverse-transform returns to real closing prices (close-only trick)
        feature_cols = RET_COLS
        close_idx = feature_cols.index('return_close')
        scaler = scalers[sector]

        def invert_close_only(arr_norm_1d):
            dummy = np.zeros((len(arr_norm_1d), len(feature_cols)))
            dummy[:, close_idx] = arr_norm_1d
            inv = scaler.inverse_transform(dummy)
            return inv[:, close_idx]

        y_pred_norm = model.predict(Xte, verbose=0).reshape(-1)
        y_test_norm = yte.reshape(-1)
        y_pred_actual_returns = invert_close_only(y_pred_norm)
        y_test_actual_returns = invert_close_only(y_test_norm)

        # Reconstruct prices: close_t ≈ close_{t-1} + diff
        predicted_prices = last_closes_test + y_pred_actual_returns
        actual_prices    = last_closes_test + y_test_actual_returns

        # Price-space metrics
        rmse_actual = np.sqrt(mean_squared_error(actual_prices, predicted_prices))
        mae_actual  = mean_absolute_error(actual_prices, predicted_prices)

        all_results[model_type][sector] = {
            'RMSE_Actual_Return': float(rmse_actual),
            'MAE_Actual_Return':  float(mae_actual),
            'actual_closing_prices': actual_prices,
            'predicted_closing_prices': predicted_prices,
        }

        # Save model
        model_path = os.path.join(OUTPUT_MODELS_DIR, f"{model_type}_model_{sector.replace(' ', '_')}.keras")
        model.save(model_path)
        print(f"{model_type} saved → {model_path} | Test MSE={test_mse:.6f} MAE={test_mae:.6f} | "
              f"Price RMSE={rmse_actual:.6f} MAE={mae_actual:.6f}")

print("\nPer-sector training complete.")



===== Sector: Engineering =====

--- Transformer ---
Epoch 1/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 6s 26ms/step - loss: 0.0799 - mae: 0.2223 - val_loss: 0.0172 - val_mae: 0.1190
Epoch 2/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0280 - mae: 0.1288 - val_loss: 0.0156 - val_mae: 0.1167
Epoch 3/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0219 - mae: 0.1145 - val_loss: 0.0083 - val_mae: 0.0806
Epoch 4/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0166 - mae: 0.0983 - val_loss: 0.0026 - val_mae: 0.0391
Epoch 5/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0130 - mae: 0.0832 - val_loss: 0.0022 - val_mae: 0.0340
Epoch 6/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0131 - mae: 0.0830 - val_loss: 0.0019 - val_mae: 0.0304
Epoch 7/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0131 - mae: 0.0847 - val_loss: 0.0046 - val_mae: 0.0594
Epoch 8/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0134 - mae: 0.0812 - val_loss: 0.0029 - val_mae: 0.0427
Epoch 9/50

RNN saved → /content/drive/MyDrive/AML /All/models_1/RNN_model_Engineering.keras | Test MSE=0.001046 MAE=0.022153 | Price RMSE=0.842759 MAE=0.577248

===== Sector: Fuel & Power =====

--- Transformer ---
Epoch 1/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step - loss: 0.2363 - mae: 0.3720 - val_loss: 0.0120 - val_mae: 0.0897
Epoch 2/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0431 - mae: 0.1613 - val_loss: 0.0089 - val_mae: 0.0712
Epoch 3/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0246 - mae: 0.1213 - val_loss: 0.0059 - val_mae: 0.0444
Epoch 4/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0179 - mae: 0.1027 - val_loss: 0.0094 - val_mae: 0.0723
Epoch 5/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0195 - mae: 0.1049 - val_loss: 0.0081 - val_mae: 0.0626
Epoch 6/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0156 - mae: 0.0910 - val_loss: 0.0063 - val_mae: 0.0460
Epoch 7/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0137 - mae: 0.0904 - val_loss: 0.00

In [ ]:
# Block 6b: Train OVERALL (pooled) models

def concat_overall(processed_data_dict, sectors):
    Xs, ys = [], []
    for s in sectors:
        if s in processed_data_dict:
            Xs.append(processed_data_dict[s]['X'])
            ys.append(ensure_targets_column(processed_data_dict[s]['y']))
    return np.concatenate(Xs, 0), np.concatenate(ys, 0)

X_all, y_all = concat_overall(processed_data_dict, sectors)
seq_len_all, n_feats_all = X_all.shape[1], X_all.shape[2]
Xtr_all, ytr_all, Xv_all, yv_all, Xte_all, yte_all = ts_split(X_all, y_all, 0.8, 0.1)
print(f"OVERALL: Train {Xtr_all.shape}, Val {Xv_all.shape}, Test {Xte_all.shape}")

model_histories_overall = {}
all_results_overall = {}

for model_type in MODEL_TYPES:
    print(f"\n=== OVERALL {model_type} ===")
    model = build_model_by_name(model_type, seq_len_all, n_feats_all)
    hist = model.fit(
        Xtr_all, ytr_all,
        validation_data=(Xv_all, yv_all),
        epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1,
        callbacks=[EARLY_STOP]
    )
    model_histories_overall[model_type] = hist

    mse, mae = model.evaluate(Xte_all, yte_all, verbose=0)
    all_results_overall[model_type] = {"Test_MSE": float(mse), "Test_MAE": float(mae)}

    path = os.path.join(OUTPUT_MODELS_DIR, f"{model_type}_model_OVERALL.keras")
    model.save(path)
    print(f"Saved OVERALL → {path} | Test MSE={mse:.6f} MAE={mae:.6f}")


OVERALL: Train (3379, 8, 5), Val (422, 8, 5), Test (423, 8, 5)

=== OVERALL Transformer ===
Epoch 1/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - loss: 0.7895 - mae: 0.4954 - val_loss: 0.0067 - val_mae: 0.0289
Epoch 2/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0209 - mae: 0.1110 - val_loss: 0.0069 - val_mae: 0.0323
Epoch 3/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0162 - mae: 0.0955 - val_loss: 0.0075 - val_mae: 0.0319
Epoch 4/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0139 - mae: 0.0878 - val_loss: 0.0077 - val_mae: 0.0329
Epoch 5/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0121 - mae: 0.0814 - val_loss: 0.0084 - val_mae: 0.0406
Epoch 6/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0110 - mae: 0.0765 - val_loss: 0.0082 - val_mae: 0.0337
Saved OVERALL → /content/drive/MyDrive/AML /All/models/Transformer_model_OVERALL.keras | Test MSE=0.002355 MAE=0.023170

=== OVERALL LSTM ===
Epoch 1/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss

In [ ]:
# Block 7: Build comparison tables (Train/Val/Test) as DataFrames & CSVs

# --- Incorporate relevant parts from Block 6b to ensure variables are defined ---
# This is done to make this cell runnable even if Block 6b wasn't executed directly before it.

def ensure_targets_column(y):
    y = np.asarray(y)
    return y.reshape(-1, 1) if y.ndim == 1 else y

def ts_split(X, y, train_ratio=0.8, val_ratio=0.1):
    n = X.shape[0]
    n_train = max(1, int(n*train_ratio))
    n_val   = max(1, int(n*val_ratio))
    n_test  = max(1, n - n_train - n_val)
    if n_train + n_val + n_test > n:
        n_test = max(1, n - n_train - n_val)
    Xtr, ytr = X[:n_train], y[:n_train]
    Xv,  yv  = X[n_train:n_train+n_val], y[n_train:n_train+n_val]
    Xte, yte = X[n_train+n_val:], y[n_train+n_val:]
    return Xtr, ytr, Xv, yv, Xte, yte

def build_model_by_name(name, seq_len, n_feats):
    if name == 'Transformer': return build_transformer_model(seq_len, n_feats)
    if name == 'LSTM':       return build_lstm_model(seq_len, n_feats)
    if name == 'RNN':        return build_rnn_model(seq_len, n_feats)
    raise ValueError(name)

EPOCHS = 50
BATCH_SIZE = 32
EARLY_STOP = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


def concat_overall(processed_data_dict, sectors):
    Xs, ys = [], []
    for s in sectors:
        if s in processed_data_dict:
            Xs.append(processed_data_dict[s]['X'])
            ys.append(ensure_targets_column(processed_data_dict[s]['y']))
    return np.concatenate(Xs, 0), np.concatenate(ys, 0)

# Re-run the OVERALL training part to define the variables
X_all, y_all = concat_overall(processed_data_dict, sectors)
seq_len_all, n_feats_all = X_all.shape[1], X_all.shape[2]

# Perform the split again if X_all is not empty
if len(X_all) > 0:
  Xtr_all, ytr_all, Xv_all, yv_all, Xte_all, yte_all = ts_split(X_all, y_all, 0.8, 0.1)
  print(f"OVERALL (re-computed): Train {Xtr_all.shape}, Val {Xv_all.shape}, Test {Xte_all.shape}")

  model_histories_overall = {}
  all_results_overall = {}

  for model_type in MODEL_TYPES:
      print(f"\n=== OVERALL {model_type} (re-training) ===")
      model = build_model_by_name(model_type, seq_len_all, n_feats_all)
      hist = model.fit(
          Xtr_all, ytr_all,
          validation_data=(Xv_all, yv_all),
          epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1,
          callbacks=[EARLY_STOP]
      )
      model_histories_overall[model_type] = hist

      mse, mae = model.evaluate(Xte_all, yte_all, verbose=0)
      all_results_overall[model_type] = {"Test_MSE": float(mse), "Test_MAE": float(mae)}

      # Save model - using the same path as Block 6b
      path = os.path.join(OUTPUT_MODELS_DIR, f"{model_type}_model_OVERALL.keras")
      model.save(path)
      print(f"Saved OVERALL (re-trained) → {path} | Test MSE={mse:.6f} MAE={mae:.6f}")
else:
    # Handle the case where X_all is empty (e.g., no sectors processed)
    print("No data to process for OVERALL model.")
    model_histories_overall = {}
    all_results_overall = {}


# --- Original code from Block 7 starts here ---

def last_or_nan(x):
    return float(x[-1]) if isinstance(x, (list, tuple)) and len(x) else (float(x) if np.ndim(x)==0 else np.nan)

def hist_to_row(hist, model_type, sector):
    h = hist.history
    return {
        "Model": model_type,
        "Sector": sector,
        "Train_MSE_last": last_or_nan(h.get('loss', [])),
        "Val_MSE_last":   last_or_nan(h.get('val_loss', [])),
        "Train_MAE_last": last_or_nan(h.get('mae', [])),
        "Val_MAE_last":   last_or_nan(h.get('val_mae', [])),
    }

# Per-sector table
rows = []
for model_type in MODEL_TYPES:
    for sector in sectors:
        # Use model_histories which should be defined from Block 6
        hist = model_histories.get(model_type, {}).get(sector)
        if hist is None:
            continue
        row = hist_to_row(hist, model_type, sector)
        # Use all_results which should be defined from Block 6
        res = all_results.get(model_type, {}).get(sector, {})
        row["Test_RMSE_Actual_Return"] = res.get("RMSE_Actual_Return", np.nan)
        row["Test_MAE_Actual_Return"]  = res.get("MAE_Actual_Return",  np.nan)
        rows.append(row)
df_per_sector = pd.DataFrame(rows).sort_values(["Model", "Sector"])
display(df_per_sector.head(12))

# OVERALL table
rows_overall = []
for model_type, hist in model_histories_overall.items():
    row = hist_to_row(hist, model_type, "OVERALL")
    test = all_results_overall.get(model_type, {})
    row["Test_MSE"] = test.get("Test_MSE", np.nan)
    row["Test_MAE"] = test.get("Test_MAE", np.nan)
    rows_overall.append(row)
df_overall = pd.DataFrame(rows_overall).sort_values(["Model"])
display(df_overall)

# Save CSVs
per_sector_csv = os.path.join(OUTPUT_RESULTS_DIR, "comparison_per_sector.csv")
overall_csv    = os.path.join(OUTPUT_RESULTS_DIR, "comparison_overall.csv")
df_per_sector.to_csv(per_sector_csv, index=False)
df_overall.to_csv(overall_csv, index=False)
print("Saved:", per_sector_csv)
print("Saved:", overall_csv)

OVERALL (re-computed): Train (3379, 8, 5), Val (422, 8, 5), Test (423, 8, 5)

=== OVERALL Transformer (re-training) ===
Epoch 1/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - loss: 0.0463 - mae: 0.1593 - val_loss: 0.0099 - val_mae: 0.0644
Epoch 2/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0117 - mae: 0.0794 - val_loss: 0.0073 - val_mae: 0.0327
Epoch 3/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0096 - mae: 0.0703 - val_loss: 0.0073 - val_mae: 0.0296
Epoch 4/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0095 - mae: 0.0699 - val_loss: 0.0082 - val_mae: 0.0356
Epoch 5/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0081 - mae: 0.0635 - val_loss: 0.0074 - val_mae: 0.0356
Epoch 6/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0078 - mae: 0.0623 - val_loss: 0.0082 - val_mae: 0.0320
Epoch 7/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0084 - mae: 0.0633 - val_loss: 0.0078 - val_mae: 0.0341
Epoch 8/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step 

,Model,Sector,Train_MSE_last,Val_MSE_last,Train_MAE_last,Val_MAE_last,Test_RMSE_Actual_Return,Test_MAE_Actual_Return
5,LSTM,Engineering,0.007770,0.002285,0.060160,0.032758,0.780778,0.515608
6,LSTM,Fuel & Power,0.009558,0.006399,0.069702,0.051481,1.658904,1.312094
7,LSTM,IT Sector,0.008394,0.004586,0.064855,0.051755,0.276484,0.191973
8,LSTM,Services & Real Estate,0.005780,0.000536,0.056911,0.020556,0.441165,0.306835
9,LSTM,Telecommunication,0.007955,0.002468,0.055460,0.031768,1.203033,0.621188
10,RNN,Engineering,0.007671,0.003036,0.061520,0.040372,0.842759,0.577248
11,RNN,Fuel & Power,0.009324,0.005762,0.067598,0.045195,1.464325,1.185529
12,RNN,IT Sector,0.008857,0.004422,0.068247,0.050619,0.278020,0.181631
13,RNN,Services & Real Estate,0.003573,0.001046,0.041115,0.027705,0.602634,0.433935
14,RNN,Telecommunication,0.008041,0.002450,0.056941,0.031305,1.113397,0.633615


,Model,Sector,Train_MSE_last,Val_MSE_last,Train_MAE_last,Val_MAE_last,Test_MSE,Test_MAE
1,LSTM,OVERALL,0.00690,0.007461,0.056770,0.040802,0.002140,0.023368
2,RNN,OVERALL,0.00656,0.007686,0.055662,0.034894,0.002307,0.026065
0,Transformer,OVERALL,0.00760,0.008274,0.060199,0.038852,0.002334,0.021772


Saved: /content/drive/MyDrive/AML /All/results_/comparison_per_sector.csv
Saved: /content/drive/MyDrive/AML /All/results_/comparison_overall.csv


In [ ]:
# Block 8: Graphs — Loss/MAE (Per-Sector & OVERALL) + Predictions & Sector Bars

def plot_loss_mae(history, title_prefix, base_name):
    # Loss
    plt.figure()
    plt.plot(history.history.get('loss', []), label='Train Loss')
    plt.plot(history.history.get('val_loss', []), label='Val Loss')
    plt.title(f"{title_prefix} — Loss")
    plt.xlabel("Epoch"); plt.ylabel("MSE"); plt.legend(); plt.tight_layout()
    p = os.path.join(OUTPUT_FIGURES_DIR, f"{base_name}_loss.png"); plt.savefig(p, dpi=160); plt.close()

    # MAE
    plt.figure()
    plt.plot(history.history.get('mae', []), label='Train MAE')
    plt.plot(history.history.get('val_mae', []), label='Val MAE')
    plt.title(f"{title_prefix} — MAE")
    plt.xlabel("Epoch"); plt.ylabel("MAE"); plt.legend(); plt.tight_layout()
    p = os.path.join(OUTPUT_FIGURES_DIR, f"{base_name}_mae.png"); plt.savefig(p, dpi=160); plt.close()

def plot_pred_vs_actual(actual, pred, title_prefix, base_name, num_points=None):
    a = actual if num_points is None else actual[-num_points:]
    p = pred   if num_points is None else pred[-num_points:]
    plt.figure()
    plt.plot(a, label="Actual")
    plt.plot(p, label="Predicted")
    plt.title(f"{title_prefix} — Prediction vs Actual")
    plt.xlabel("Time"); plt.ylabel("Price"); plt.legend(); plt.tight_layout()
    path = os.path.join(OUTPUT_FIGURES_DIR, f"{base_name}_pred_vs_actual.png")
    plt.savefig(path, dpi=160); plt.close()

def plot_zoom(actual, pred, title_prefix, base_name, start=-150, end=None):
    a = actual[start:end]; p = pred[start:end]
    plt.figure()
    plt.plot(a, label="Actual"); plt.plot(p, label="Predicted")
    plt.title(f"{title_prefix} — Zoom")
    plt.xlabel("Time"); plt.ylabel("Price"); plt.legend(); plt.tight_layout()
    path = os.path.join(OUTPUT_FIGURES_DIR, f"{base_name}_zoom.png")
    plt.savefig(path, dpi=160); plt.close()

def bar_by_sector(metric_name, model_type, results_dict, suffix):
    sectors_list, vals = [], []
    for s in sectors:
        if s in results_dict.get(model_type, {}):
            sectors_list.append(s)
            vals.append(results_dict[model_type][s][metric_name])
    if not sectors_list: return
    plt.figure(figsize=(10,4.5))
    plt.bar(range(len(vals)), vals)
    plt.xticks(range(len(vals)), sectors_list, rotation=45, ha='right')
    plt.ylabel(metric_name.replace('_', ' ')); plt.title(f"{model_type} — {metric_name} across sectors")
    plt.tight_layout()
    path = os.path.join(OUTPUT_FIGURES_DIR, f"{model_type}_{suffix}_{metric_name}.png")
    plt.savefig(path, dpi=160); plt.close()

# Per-sector: curves and predictions
for model_type in MODEL_TYPES:
    for s in sectors:
        hist = model_histories.get(model_type, {}).get(s)
        if hist is None:
            continue
        base = f"{model_type}_{s.replace(' ', '_')}"
        plot_loss_mae(hist, f"{model_type} | {s}", base)

        res = all_results.get(model_type, {}).get(s, {})
        predicted_prices = res.get('predicted_closing_prices')
        actual_prices = res.get('actual_closing_prices')

        if predicted_prices is not None and actual_prices is not None and len(predicted_prices) > 0:
            plot_pred_vs_actual(actual_prices, predicted_prices,
                                f"{model_type} | {s}", base)
            plot_zoom(actual_prices, predicted_prices,
                      f"{model_type} | {s}", base)

# OVERALL: curves
for model_type, hist in model_histories_overall.items():
    base = f"{model_type}_OVERALL"
    plot_loss_mae(hist, f"{model_type} | OVERALL", base)

# Aggregate bars across sectors
for model_type in MODEL_TYPES:
    bar_by_sector('RMSE_Actual_Return', model_type, all_results, 'Test')
    bar_by_sector('MAE_Actual_Return',  model_type, all_results, 'Test')

print("Saved: per-sector Loss/MAE, OVERALL Loss/MAE, per-sector Pred vs Actual (with zoom), and sector bar charts.")

Saved: per-sector Loss/MAE, OVERALL Loss/MAE, per-sector Pred vs Actual (with zoom), and sector bar charts.


In [ ]:
# Block 9: Utility — quick glance

print("Comparison CSVs:")
print(os.path.join(OUTPUT_RESULTS_DIR, "comparison_per_sector.csv"))
print(os.path.join(OUTPUT_RESULTS_DIR, "comparison_overall.csv"))

print("\nExample figure paths:")
for m in MODEL_TYPES:
    print(os.path.join(OUTPUT_FIGURES_DIR, f"{m}_OVERALL_loss.png"))
    print(os.path.join(OUTPUT_FIGURES_DIR, f"{m}_OVERALL_mae.png"))


Comparison CSVs:
/content/drive/MyDrive/AML /All/results/comparison_per_sector.csv
/content/drive/MyDrive/AML /All/results/comparison_overall.csv

Example figure paths:
/content/drive/MyDrive/AML /All/figures/Transformer_OVERALL_loss.png
/content/drive/MyDrive/AML /All/figures/Transformer_OVERALL_mae.png
/content/drive/MyDrive/AML /All/figures/LSTM_OVERALL_loss.png
/content/drive/MyDrive/AML /All/figures/LSTM_OVERALL_mae.png
/content/drive/MyDrive/AML /All/figures/RNN_OVERALL_loss.png
/content/drive/MyDrive/AML /All/figures/RNN_OVERALL_mae.png


In [ ]:
'''
# Block 10: Real-world testing on unseen market data (paper-style)

# Put a fresh CSV with columns:
#   sector, trading date, open, high, low, close, volume
REAL_WORLD_CSV   = '.csv'  # update with your unseen data file
REAL_RESULTS_CSV = os.path.join(OUTPUT_RESULTS_DIR, "realworld_comparison.csv")
REAL_FIG_DIR     = os.path.join(OUTPUT_FIGURES_DIR, "realworld")
os.makedirs(REAL_FIG_DIR, exist_ok=True)

def _normalize_columns(df_):
    df_ = df_.copy()
    df_.columns = df_.columns.str.strip()
    if 'trading date' not in df_.columns:
        for k in ['trading_date', 'date', 'Trading Date', 'Trading date']:
            if k in df_.columns:
                df_['trading date'] = df_[k]
                break
    df_['trading date'] = pd.to_datetime(df_['trading date'], errors='coerce')
    df_ = df_.dropna(subset=['trading date']).sort_values(['sector', 'trading date'])
    return df_

def _make_stationary_returns(g):
    g = g.copy()
    smooth = g[FEATURES].rolling(window=10, min_periods=1).mean()
    for c in FEATURES:
        g[f'return_{c}'] = smooth[c].diff()
    g = g.dropna().reset_index(drop=True)
    return g

def _apply_training_scaler(g, sector_name):
    pkl = os.path.join(SCALERS_DIR, f"scaler_{sector_name.replace(' ', '_')}.pkl")
    if not os.path.exists(pkl):
        raise FileNotFoundError(f"Saved scaler not found for sector {sector_name}: {pkl}")
    scaler = joblib.load(pkl)
    g[RET_COLS] = scaler.transform(g[RET_COLS])
    return g, scaler

def _create_sequences_for_inference(data_df, seq_len=SEQ_LEN):
    X, last_closes, idx_map = [], [], []
    for i in range(seq_len, len(data_df)):
        X.append(data_df[RET_COLS].iloc[i-seq_len:i].values)
        last_closes.append(data_df['close'].iloc[i-1])
        idx_map.append(i)
    return np.array(X), np.array(last_closes), np.array(idx_map)

def _reconstruct_prices(last_closes, pred_returns, true_returns):
    pred_prices = last_closes + pred_returns
    true_prices = last_closes + true_returns
    return pred_prices, true_prices

def _predict_with_model(model_path, X):
    model = tf.keras.models.load_model(model_path, compile=False)
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    y_pred = model.predict(X, verbose=0).reshape(-1)
    return y_pred

def _plot_realworld(actual_prices, predicted_prices, title, base):
    # Full
    plt.figure()
    plt.plot(actual_prices, label="Actual")
    plt.plot(predicted_prices, label="Predicted")
    plt.title(f"{title} — Real-world")
    plt.xlabel("Time"); plt.ylabel("Price"); plt.legend(); plt.tight_layout()
    p = os.path.join(REAL_FIG_DIR, f"{base}_realworld.png")
    plt.savefig(p, dpi=160); plt.close()

    # Zoom (tail)
    z = min(150, len(actual_prices))
    plt.figure()
    plt.plot(actual_prices[-z:], label="Actual")
    plt.plot(predicted_prices[-z:], label="Predicted")
    plt.title(f"{title} — Real-world (zoom last {z})")
    plt.xlabel("Time"); plt.ylabel("Price"); plt.legend(); plt.tight_layout()
    p = os.path.join(REAL_FIG_DIR, f"{base}_realworld_zoom.png")
    plt.savefig(p, dpi=160); plt.close()

# Load real-world data
real_df = pd.read_csv(REAL_WORLD_CSV)
real_df = _normalize_columns(real_df)

rows = []

# Per-sector real-world test
for sector in sectors:
    sec_df = real_df[real_df['sector'] == sector].copy()
    if len(sec_df) < SEQ_LEN + 5:
        print(f"[Real] Skipping {sector}: too few rows ({len(sec_df)})")
        continue

    sec_proc = _make_stationary_returns(sec_df)
    try:
        sec_proc, scaler = _apply_training_scaler(sec_proc, sector)
    except FileNotFoundError as e:
        print(e)
        continue

    X_inf, last_closes_inf, idx_map = _create_sequences_for_inference(sec_proc, SEQ_LEN)
    y_true_norm = sec_proc['return_close'].iloc[idx_map].values

    close_idx = RET_COLS.index('return_close')
    def invert_close_only(arr_norm_1d):
        dummy = np.zeros((len(arr_norm_1d), len(RET_COLS)))
        dummy[:, close_idx] = arr_norm_1d
        inv = scaler.inverse_transform(dummy)
        return inv[:, close_idx]

    y_true_ret = invert_close_only(y_true_norm)

    for model_type in MODEL_TYPES:
        model_path = os.path.join(OUTPUT_MODELS_DIR, f"{model_type}_model_{sector.replace(' ', '_')}.keras")
        if not os.path.exists(model_path):
            print(f"[Real] Missing model: {model_path}")
            continue

        y_pred_norm = _predict_with_model(model_path, X_inf)
        y_pred_ret  = invert_close_only(y_pred_norm)

        pred_px, true_px = _reconstruct_prices(last_closes_inf, y_pred_ret, y_true_ret)
        rmse = np.sqrt(mean_squared_error(true_px, pred_px))
        mae  = mean_absolute_error(true_px, pred_px)

        rows.append({
            "Scope": "Per-Sector",
            "Sector": sector,
            "Model": model_type,
            "RMSE_RealPrice": float(rmse),
            "MAE_RealPrice":  float(mae),
            "N": len(true_px)
        })

        base = f"{model_type}_{sector.replace(' ', '_')}"
        _plot_realworld(true_px, pred_px, f"{model_type} | {sector}", base)

# OVERALL real-world test (pooled)
X_all, y_all_ret, last_all, pairs = [], [], [], []
for sector in sectors:
    sec_df = real_df[real_df['sector'] == sector].copy()
    if len(sec_df) < SEQ_LEN + 5:
        continue
    sec_proc = _make_stationary_returns(sec_df)
    try:
        sec_proc, scaler = _apply_training_scaler(sec_proc, sector)
    except FileNotFoundError:
        continue
    X_inf, last_inf, idx_map = _create_sequences_for_inference(sec_proc, SEQ_LEN)
    y_true_norm = sec_proc['return_close'].iloc[idx_map].values

    close_idx = RET_COLS.index('return_close')
    def inv_close(arr_norm_1d):
        dummy = np.zeros((len(arr_norm_1d), len(RET_COLS)))
        dummy[:, close_idx] = arr_norm_1d
        inv = scaler.inverse_transform(dummy)
        return inv[:, close_idx]

    y_true_ret = inv_close(y_true_norm)

    X_all.append(X_inf); y_all_ret.append(y_true_ret); last_all.append(last_inf)
    pairs.append((sector, len(X_inf)))

if len(X_all):
    X_all = np.concatenate(X_all, axis=0)
    y_all_ret = np.concatenate(y_all_ret, axis=0)
    last_all  = np.concatenate(last_all, axis=0)

    for model_type in MODEL_TYPES:
        overall_path = os.path.join(OUTPUT_MODELS_DIR, f"{model_type}_model_OVERALL.keras")
        if not os.path.exists(overall_path):
            print(f"[Real] Missing OVERALL model: {overall_path}")
            continue

        y_pred_norm = _predict_with_model(overall_path, X_all)

        # Convert predicted returns into actual units per-window using each sector's scaler
        pred_ret_actual = np.empty_like(y_pred_norm, dtype=float)
        cursor = 0
        for sector, count in pairs:
            scaler = joblib.load(os.path.join(SCALERS_DIR, f"scaler_{sector.replace(' ', '_')}.pkl"))
            seg = y_pred_norm[cursor:cursor+count]
            close_idx = RET_COLS.index('return_close')
            dummy = np.zeros((len(seg), len(RET_COLS)))
            dummy[:, close_idx] = seg
            inv = scaler.inverse_transform(dummy)
            pred_ret_actual[cursor:cursor+count] = inv[:, close_idx]
            cursor += count

        pred_px, true_px = _reconstruct_prices(last_all, pred_ret_actual, y_all_ret)
        rmse = np.sqrt(mean_squared_error(true_px, pred_px))
        mae  = mean_absolute_error(true_px, pred_px)

        rows.append({
            "Scope": "OVERALL",
            "Sector": "ALL",
            "Model": model_type,
            "RMSE_RealPrice": float(rmse),
            "MAE_RealPrice":  float(mae),
            "N": len(true_px)
        })

        tail = min(1000, len(true_px))
        plt.figure()
        plt.plot(true_px[-tail:], label="Actual")
        plt.plot(pred_px[-tail:], label="Predicted")
        plt.title(f"{model_type} | OVERALL — Real-world (tail {tail})")
        plt.xlabel("Time"); plt.ylabel("Price"); plt.legend(); plt.tight_layout()
        p = os.path.join(REAL_FIG_DIR, f"{model_type}_OVERALL_realworld_tail.png")
        plt.savefig(p, dpi=160); plt.close()

# Save the real-world comparison table
df_realworld = pd.DataFrame(rows).sort_values(["Scope", "Model", "Sector"])
df_realworld.to_csv(REAL_RESULTS_CSV, index=False)
print("Saved real-world comparison:", REAL_RESULTS_CSV)
display(df_realworld.head(20))
'''


'\n# Block 10: Real-world testing on unseen market data (paper-style)\n\n# Put a fresh CSV with columns:\n#   sector, trading date, open, high, low, close, volume\nREAL_WORLD_CSV   = \'.csv\'  # update with your unseen data file\nREAL_RESULTS_CSV = os.path.join(OUTPUT_RESULTS_DIR, "realworld_comparison.csv")\nREAL_FIG_DIR     = os.path.join(OUTPUT_FIGURES_DIR, "realworld")\nos.makedirs(REAL_FIG_DIR, exist_ok=True)\n\ndef _normalize_columns(df_):\n    df_ = df_.copy()\n    df_.columns = df_.columns.str.strip()\n    if \'trading date\' not in df_.columns:\n        for k in [\'trading_date\', \'date\', \'Trading Date\', \'Trading date\']:\n            if k in df_.columns:\n                df_[\'trading date\'] = df_[k]\n                break\n    df_[\'trading date\'] = pd.to_datetime(df_[\'trading date\'], errors=\'coerce\')\n    df_ = df_.dropna(subset=[\'trading date\']).sort_values([\'sector\', \'trading date\'])\n    return df_\n\ndef _make_stationary_returns(g):\n    g = g.copy()\n 

In [ ]:
# Block 11: Paper-style Processed vs Raw plots (per sector & model

PAPER_FIG_DIR = os.path.join(OUTPUT_FIGURES_DIR, "paper_style")
os.makedirs(PAPER_FIG_DIR, exist_ok=True)

def _reload_model(model_type, sector):
    path = os.path.join(OUTPUT_MODELS_DIR, f"{model_type}_model_{sector.replace(' ', '_')}.keras")
    if not os.path.exists(path):
        print(f"[Block11] Missing model file: {path}")
        return None, path
    # Allow unsafe deserialization for the Lambda layer with a Python lambda
    m = tf.keras.models.load_model(path, compile=False, safe_mode=False)
    m.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return m, path

def _sector_split_again(sector):
    """Recreate the exact 80/10/10 split used in training for this sector."""
    pack = processed_data_dict[sector]
    X = pack['X']
    y = pack['y'].reshape(-1, 1) if np.ndim(pack['y']) == 1 else pack['y']
    Xtr, ytr, Xv, yv, Xte, yte = ts_split(X, y, 0.8, 0.1)
    last_closes = pack['last_closes']
    last_closes_test = last_closes[len(ytr) + len(yv):]
    scaler = scalers[sector]
    return Xtr, ytr, Xv, yv, Xte, yte, last_closes_test, scaler

def _invert_close_only(scaler, ret_norm):
    """Invert only the close-return dimension back to actual units."""
    close_idx = RET_COLS.index('return_close')
    dummy = np.zeros((len(ret_norm), len(RET_COLS)))
    dummy[:, close_idx] = ret_norm
    inv = scaler.inverse_transform(dummy)
    return inv[:, close_idx]

def _plot_processed_returns(y_true_norm, y_pred_norm, title_prefix, base):
    # Full processed comparison (normalized returns)
    plt.figure()
    plt.plot(y_true_norm, label="True (processed)")
    plt.plot(y_pred_norm, label="Pred (processed)")
    plt.title(f"{title_prefix} — Processed (returns)")
    plt.xlabel("Test index"); plt.ylabel("Normalized return")
    plt.legend(); plt.tight_layout()
    p = os.path.join(PAPER_FIG_DIR, f"{base}_processed_returns.png")
    plt.savefig(p, dpi=170); plt.close()

    # Zoom tail
    z = min(150, len(y_true_norm))
    plt.figure()
    plt.plot(y_true_norm[-z:], label="True (processed)")
    plt.plot(y_pred_norm[-z:], label="Pred (processed)")
    plt.title(f"{title_prefix} — Processed (returns) zoom last {z}")
    plt.xlabel("Test index (tail)"); plt.ylabel("Normalized return")
    plt.legend(); plt.tight_layout()
    p = os.path.join(PAPER_FIG_DIR, f"{base}_processed_returns_zoom.png")
    plt.savefig(p, dpi=170); plt.close()

def _plot_raw_prices(actual_px, pred_px, title_prefix, base):
    # Full raw price comparison
    plt.figure()
    plt.plot(actual_px, label="Actual price")
    plt.plot(pred_px, label="Predicted price")
    plt.title(f"{title_prefix} — Raw closing price")
    plt.xlabel("Test index"); plt.ylabel("Price")
    plt.legend(); plt.tight_layout()
    p = os.path.join(PAPER_FIG_DIR, f"{base}_raw_prices.png")
    plt.savefig(p, dpi=170); plt.close()

    # Zoom tail
    z = min(150, len(actual_px))
    plt.figure()
    plt.plot(actual_px[-z:], label="Actual price")
    plt.plot(pred_px[-z:], label="Predicted price")
    plt.title(f"{title_prefix} — Raw price (zoom last {z})")
    plt.xlabel("Test index (tail)"); plt.ylabel("Price")
    plt.legend(); plt.tight_layout()
    p = os.path.join(PAPER_FIG_DIR, f"{base}_raw_prices_zoom.png")
    plt.savefig(p, dpi=170); plt.close()

# --- Generate paired plots for every (model × sector) ---
for model_type in MODEL_TYPES:
    for sector in sectors:
        if sector not in processed_data_dict:
            continue

        # Reload model & recreate split
        model, model_path = _reload_model(model_type, sector)
        if model is None:
            continue
        Xtr, ytr, Xv, yv, Xte, yte, last_closes_test, scaler = _sector_split_again(sector)

        # Predict in processed (normalized) space
        y_pred_norm = model.predict(Xte, verbose=0).reshape(-1)
        y_true_norm = yte.reshape(-1)

        # Processed (normalized returns) plots — paper Fig. 9 analogue
        base = f"{model_type}_{sector.replace(' ', '_')}"
        _plot_processed_returns(y_true_norm, y_pred_norm, f"{model_type} | {sector}", base)

        # Convert to real return units and reconstruct raw price — paper Figs. 10–13 analogue
        y_pred_ret = _invert_close_only(scaler, y_pred_norm)
        y_true_ret = _invert_close_only(scaler, y_true_norm)
        pred_px = last_closes_test + y_pred_ret
        true_px = last_closes_test + y_true_ret

        _plot_raw_prices(true_px, pred_px, f"{model_type} | {sector}", base)

        # (Optional) price metrics for quick console feedback
        rmse = np.sqrt(mean_squared_error(true_px, pred_px))
        mae  = mean_absolute_error(true_px, pred_px)
        print(f"[Block11] {model_type} | {sector}: price RMSE={rmse:.6f} MAE={mae:.6f}")

print(f"\nBlock 11 done. Paper-style figures saved under: {PAPER_FIG_DIR}")

[Block11] Transformer | Engineering: price RMSE=0.847842 MAE=0.526014
[Block11] Transformer | Fuel & Power: price RMSE=1.488694 MAE=1.228299
[Block11] Transformer | IT Sector: price RMSE=0.330997 MAE=0.208024
[Block11] Transformer | Services & Real Estate: price RMSE=0.526775 MAE=0.360958
[Block11] Transformer | Telecommunication: price RMSE=1.196357 MAE=0.694014
[Block11] LSTM | Engineering: price RMSE=0.770392 MAE=0.491081
[Block11] LSTM | Fuel & Power: price RMSE=1.571247 MAE=1.247905
[Block11] LSTM | IT Sector: price RMSE=0.279854 MAE=0.206141
[Block11] LSTM | Services & Real Estate: price RMSE=0.450916 MAE=0.321860
[Block11] LSTM | Telecommunication: price RMSE=1.129828 MAE=0.572693
[Block11] RNN | Engineering: price RMSE=0.819720 MAE=0.520160
[Block11] RNN | Fuel & Power: price RMSE=1.326326 MAE=1.069052
[Block11] RNN | IT Sector: price RMSE=0.284075 MAE=0.182168
[Block11] RNN | Services & Real Estate: price RMSE=1.112495 MAE=0.844186
[Block11] RNN | Telecommunication: price RMSE

# Result Analysis